# 额外的周末练习 —— 第 2 周

## 练习目标（理念）

用第 2 周学到的能力，把第 1 周的「技术问答」做成可演示的完整原型：

- **Gradio UI**：浏览器里聊天
- **流式（streaming）**：边生成边显示
- **系统提示（system prompt）**：注入领域专家人设
- **模型切换**：在多个后端模型间选择
- **加分项**：演示工具调用（tool use）；更大胆可加语音输入/输出

本作业示例做成「独立游戏设计问答助手」：带设计术语工具、OpenRouter 上的 GPT，以及经 OpenRouter 路由的 Llama。

## 怎么跑

1. `.env` 配置 `OPENROUTER_API_KEY`
2. 从上到下运行代码格，最后一格 `app.launch()` 打开界面
3. 在下拉框切换模型，向助手提问游戏设计相关问题


In [1]:
# ========== 导入依赖：后面 UI、密钥、API 都靠这些 ==========
# Imports

# 导入 os：读环境变量（例如 OPENROUTER_API_KEY）
import os
# 导入 json：解析工具调用（tool call）返回的 arguments JSON
import json
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：OpenAI 兼容客户端（本作业指向 OpenRouter）
from openai import OpenAI
# 导入 gradio：快速搭聊天 Web UI
import gradio as gr


In [2]:
# ========== 常量：模型名与 API 基址集中配置 ==========
# constants

# OpenAI 侧常用小模型名（本作业 UI 标签也会用到相近命名）
MODEL_GPT = 'gpt-4o-mini'
# 本地/路由侧 Llama 模型短名
MODEL_LLAMA = 'llama3.2'
# OpenRouter Chat Completions 的 base_url（勿改）
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
# 本机 Ollama 的 OpenAI 兼容地址（本作业客户端主要走 OpenRouter，此常量保留对照）
OLLAMA_BASE_URL = 'http://localhost:11434/v1'


In [3]:
# ========== 环境与客户端：读密钥、建 OpenRouter 客户端 ==========
# set up environment

# 加载 .env；override=True 用文件覆盖已有环境变量
load_dotenv(override=True)
# 读取 OpenRouter 密钥（变量名必须是 OPENROUTER_API_KEY）
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# OpenAI客户端
# OpenAI client
# 指向 OpenRouter：后续所有 chat.completions 都经此 client
openai_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

# 型号选择：仅 gpt-4o-mini 和 Llama（OpenRouter 型号 ID）
# Model choices: gpt-4o-mini and Llama only (OpenRouter model IDs)
# UI 显示名 -> OpenRouter 路由 model id
MODELS = {
    "gpt-4o-mini": "openai/gpt-4o-mini",
    "Llama": "ollama/llama3.2",
}

# 下拉框默认选中的显示名
DEFAULT_MODEL = "gpt-4o-mini"


In [4]:
# ========== 系统提示：把助手塑造成独立游戏设计专家 ==========
# SYSTEM_PROMPT 原文影响模型行为，必须保持英文原样，不要翻译

SYSTEM_PROMPT = """
You are an expert indie game designer and technical game architect.

Your role is to generate practical, implementable game ideas for a solo developer.

Guidelines:
- Focus on core gameplay mechanics and game loops.
- Avoid vague descriptions.
- Keep art requirements minimal unless explicitly requested.
- Suggest systems that are technically feasible for a small team or solo developer.
- When appropriate, describe how the game could be structured in code (systems, state management, logic flow).
- Prioritize replayability and retention.
- Be concise but clear.
- Use structured sections when explaining ideas.

Always structure your answers like this:

1. Game Concept (1–2 sentences)
2. Core Loop
3. Core Mechanics
4. Progression System
5. Technical Implementation Notes
6. Why This Is Good For a Solo Developer
"""


In [5]:
# ========== 设计术语工具：给模型可查询的短笔记 ==========
# 设计参考：模型在讨论术语时可以提取的简短注释
# Design reference: short notes the model can pull when discussing a term

# 术语小写键 -> 英文说明字符串（作为 tool 返回内容，勿改文案）
DESIGN_REF = {
    "core loop": "Core loop: the repeatable cycle of actions that defines the game (e.g. explore → fight → loot → upgrade → repeat). Keep it tight for solo dev.",
    "roguelike": "Roguelike: permadeath, procedural levels, often turn-based. Great for replayability and low asset count; focus on systems over content.",
    "metroidvania": "Metroidvania: interconnected world, ability-gated progression, backtracking. Plan your map and lock/key abilities early.",
    "procgen": "Procedural generation: levels or content from algorithms. Reduces art burden; balance with seeds and tuning knobs for feel.",
    "state machine": "State machine: game or entity has discrete states (idle, attack, hurt). Clean for AI and animation; easy to implement and debug.",
    "game jam": "Game jam: short, scoped event (e.g. 48h). Constrains scope and forces a clear core loop—ideal for solo practice.",
    "retention": "Retention: what brings players back (daily goals, meta-progression, unlocks). Even small games benefit from a simple progression hook.",
    "solo developer": "Solo dev: scope small, reuse systems, avoid custom art where possible. Prototype the core loop first; cut everything else.",
}

def get_design_ref(term: str) -> str:
    # 规范化查询词：小写并去首尾空白，便于字典查找
    key = (term or "").lower().strip()
    # 命中则返回缓存说明；未命中则返回提示模型自行简短发挥的英文兜底句
    return DESIGN_REF.get(key, f"No cached note for '{term}'. Rely on your expertise and keep it brief for a solo dev.")

def design_ref_tool():
    """OpenAI-style tool spec for the design reference lookup."""
    # 返回 OpenAI tools 协议里的 function 描述，供 chat.completions 的 tools= 参数使用
    return {
        "type": "function",
        "function": {
            "name": "get_design_ref",
            "description": "Fetch a short reference note for a game design term (core loop, roguelike, metroidvania, procgen, state machine, game jam, retention, solo developer). Use to back up your answer when relevant.",
            "parameters": {
                "type": "object",
                "properties": {"term": {"type": "string", "description": "Game design term to look up"}},
                "required": ["term"],
                "additionalProperties": False,
            },
        },
    }

# 当前启用的工具列表：只有设计术语查询这一个
TOOLS = [design_ref_tool()]


In [6]:
# ========== 工具调用辅助：把 API 消息格式化并真正执行 tool ==========

def assistant_as_api_message(msg):
    """Turn the assistant message (with optional tool_calls) into the dict format the API expects."""
    # 把 SDK 返回的 assistant message 转成下一轮请求可追加的 dict
    return {
        "role": "assistant",
        # content 可能为 None（纯 tool_calls 时），用空串兜底
        "content": msg.content or "",
        # 逐条展开 tool_calls：保留 id、type、function.name/arguments
        "tool_calls": [
            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ],
    }

def run_tools_and_collect(assistant_msg):
    """Run any tool calls in the assistant message and return the list of tool response dicts."""
    # 收集每条 tool 的 role=tool 回复，稍后 extend 进 messages
    out = []
    # 遍历模型请求的每一个 tool_call
    for tc in assistant_msg.tool_calls:
        # 要调用的函数名
        name = tc.function.name
        try:
            # arguments 一般是 JSON 字符串；解析失败则用空 dict
            args = json.loads(tc.function.arguments or "{}")
        except json.JSONDecodeError:
            args = {}
        if name == "get_design_ref":
            # 已知工具：查设计术语笔记
            text = get_design_ref(args.get("term", ""))
        else:
            # 未知工具名：返回英文错误说明（保持原字符串）
            text = f"Unknown tool: {name}"
        # 按协议带上 tool_call_id，让模型知道这是哪次调用的结果
        out.append({"role": "tool", "content": text, "tool_call_id": tc.id})
    return out


In [7]:
# ========== 对话编排：组 messages、工具循环、流式输出 ==========

def build_conversation(history, latest_user_text):
    """Conversation as API messages: system + history + latest user."""
    # 把 Gradio 历史条目压成 role/content 扁平列表
    flat = [{"role": m.get("role", "user"), "content": m.get("content", "") or ""} for m in (history or [])]
    # 前缀 system prompt，后缀本轮用户输入
    return [{"role": "system", "content": SYSTEM_PROMPT}] + flat + [{"role": "user", "content": latest_user_text}]

def _yield_in_chunks(full_text, chunk_size=8):
    """Yield full_text in small chunks so the UI can show progressive updates."""
    # 累加器：模拟流式，让无原生 stream 的工具路径也能渐进刷新 UI
    acc = ""
    # 按 chunk_size 切片递增 yield
    for i in range(0, len(full_text), chunk_size):
        acc = full_text[: i + chunk_size]
        yield acc
    # 边界兜底：若循环未覆盖全文再 yield 一次完整文本
    if len(acc) < len(full_text):
        yield full_text

def reply_with_tools(model_id, messages):
    """Call model with tools; run tool calls and re-call until we get a text reply. Yields in chunks so UI streams."""
    # 第一次调用：带上 TOOLS，允许模型发起 tool_calls
    resp = openai_client.chat.completions.create(model=model_id, messages=messages, tools=TOOLS)
    # 若 finish_reason 仍是 tool_calls，就执行工具并再请求，直到拿到文本答复
    while getattr(resp.choices[0], "finish_reason", None) == "tool_calls":
        msg = resp.choices[0].message
        # 把带 tool_calls 的 assistant 消息写回对话
        messages.append(assistant_as_api_message(msg))
        # 追加各 tool 的执行结果
        messages.extend(run_tools_and_collect(msg))
        # 用更新后的 messages 再调一次
        resp = openai_client.chat.completions.create(model=model_id, messages=messages, tools=TOOLS)
    # 最终文本（可能为空串）
    content = resp.choices[0].message.content or ""
    # 切块 yield，驱动 Gradio 渐进显示
    for chunk in _yield_in_chunks(content):
        yield chunk

def reply_streaming(model_id, messages):
    """Stream model reply using OpenAI-compatible streaming (OpenRouter)."""
    # 开启 stream=True，逐 token/片段推送
    stream = openai_client.chat.completions.create(
        model=model_id, messages=messages, stream=True
    )
    # 累积已收到的文本
    acc = ""
    for chunk in stream:
        # 某些心跳 chunk 可能没有 choices，跳过
        if not chunk.choices:
            continue
        # delta.content 是本片段新增文本
        part = chunk.choices[0].delta.content
        if part:
            acc += part
            # 每次 yield 完整累计串，方便 Chatbot 直接替换显示
            yield acc

def generate_reply(user_text, history, model_name):
    """Single entry point: build messages, pick model, then either run tools or stream."""
    # 组装 system + 历史 + 本轮用户
    messages = build_conversation(history, user_text)
    # 显示名映射到 OpenRouter model id；未知则回退默认
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    # ollama/ 前缀走纯流式；其他（如 openai/...）走工具循环
    use_tools = not model_id.startswith("ollama/")

    if use_tools:
        # GPT 路径：支持 function calling
        for content in reply_with_tools(model_id, messages):
            yield content
    else:
        # Llama 路径：直接 streaming，不带 tools
        for content in reply_streaming(model_id, messages):
            yield content


In [8]:
# ========== UI：助手函数与 Gradio Blocks 布局 ==========
# UI: helpers and layout

def as_display_msg(item):
    """Normalize a history item to {role, content} for the chatbot."""
    # 已是 dict：取出 role/content；否则整段当 user 文本
    if isinstance(item, dict):
        return {"role": item.get("role", "user"), "content": item.get("content", "") or ""}
    return {"role": "user", "content": str(item)}

def append_question_and_clear(question, chat_history):
    """Add the user question to chat history and clear the input. Returns (empty input, updated history)."""
    # 复制历史，避免原地改到共享状态出问题
    chat_history = list(chat_history or [])
    if question:
        # 把用户问题追加为 messages 格式
        chat_history.append({"role": "user", "content": question})
    # 返回：清空输入框 + 更新后的历史
    return "", chat_history

def stream_reply_into_history(chat_history, model_name):
    """Take the last user message from history, get model reply, yield full history + growing assistant text."""
    chat_history = list(chat_history or [])
    # 空历史直接结束 generator
    if not chat_history:
        return
    # 规范化最后一条
    last = as_display_msg(chat_history[-1])
    # 若最后一条不是 user，说明状态异常，不生成
    if last["role"] != "user":
        return
    # 本轮用户文本
    user_text = last["content"]
    # 此前各轮（不含本轮 user）作为 API history
    prev_turns = [as_display_msg(h) for h in chat_history[:-1]]
    # 界面上已展示的完整历史（含本轮 user）
    displayed = [as_display_msg(h) for h in chat_history]
    # 边生成边 yield：displayed + 不断变长的 assistant 消息
    for content in generate_reply(user_text, prev_turns, model_name):
        yield displayed + [{"role": "assistant", "content": content}]

# 用 Blocks 搭页面：标题写在构造参数里
with gr.Blocks(title="Indie Game Design Q&A") as app:
    # 页面说明 Markdown（UI 文案原文保留）
    gr.Markdown("Ask about game design, mechanics, or solo dev. Try: *Design a roguelike* or *Core loop for a puzzle game*.")
    # 聊天窗口：type=messages 使用 role/content 列表
    chat_log = gr.Chatbot(height=500, type="messages")
    # 模型下拉：选项来自 MODELS 的键
    model_selector = gr.Dropdown(choices=list(MODELS.keys()), value=DEFAULT_MODEL, label="Model")
    with gr.Row():
        # 问题输入框占较宽比例
        question_input = gr.Textbox(label="Your question", placeholder="Ask a game design question...", scale=4)
        # 清空按钮
        reset_btn = gr.Button("Clear", scale=1)
    # 主提交按钮
    send_btn = gr.Button("Submit", variant="primary")

    def reset_chat():
        # 清空聊天记录与输入框
        return [], ""

    # 回车：先追加用户消息并清空输入，再流式写入助手回复
    question_input.submit(append_question_and_clear, [question_input, chat_log], [question_input, chat_log]).then(
        stream_reply_into_history, [chat_log, model_selector], chat_log
    )
    # 点击 Submit：同一条事件链
    send_btn.click(append_question_and_clear, [question_input, chat_log], [question_input, chat_log]).then(
        stream_reply_into_history, [chat_log, model_selector], chat_log
    )
    # Clear：重置 chatbot 与文本框
    reset_btn.click(reset_chat, None, [chat_log, question_input])


In [ ]:
# ========== 启动 Gradio 应用 ==========
# launch() 会起本地 Web 服务并打印访问地址；在笔记本里通常直接内嵌或开新标签
app.launch()
